# Step 6: Second-Level (Micro-Category) Clustering

This notebook performs micro-category clustering within each macro-category, classifies proposals into sub-clusters, and computes silhouette scores.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"
cluster_subclusters_fp =  f"{DATA_OUT_BASE_PATH}7.cluster_subcluster.json"
second_level_classified_proposals_fp =  f"{DATA_OUT_BASE_PATH}7.second_level_classified_proposals.json"
second_level_classified_proposals_errors_fp = f"{DATA_OUT_BASE_PATH}7.second_level_classified_proposals_errors.json"
second_level_clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}7.second_level_clusterized_proposals.json"

# Analysis out
cluster_summaries_fp = f"{DATA_OUT_BASE_PATH}7.second_level_cluster_summaries.json"
analysis_results_fp = f"{DATA_OUT_BASE_PATH}7.second_level_results.json"
filtered_clusters_fp = f"{DATA_OUT_BASE_PATH}7.second_level_filtered_clusters.json"

In [ ]:
clusterized_proposals = json.load(open(clusterized_proposals_fp))

## Divide proposals in chunks and build clusters for each chunk

In [ ]:
prompt_cluster = """The task is to cluster a list of articles.
In order to design the clusters, please consider the most relevant and emerging topics mentioned whithin articles.

Provide in output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    
Output:
    List[Cluster] # the list of cluster definition
    """

## Run cluster creation

In [ ]:
def chunked_iterator(l, chunk_size=100, overlap=50):
    """Yield chunks of the list with specified overlap."""
    if overlap >= chunk_size:
        raise ValueError("Overlap must be less than the chunk size.")

    # Iterate over the list in steps of `chunk_size - overlap`
    step = chunk_size - overlap
    for start in range(0, len(l), step):
        if start < len(l)-1 and len(l[start:start + chunk_size]) > overlap:
            yield l[start:start + chunk_size]

In [ ]:
cluster_title_to_chunks = {}
for cluster_title, cluster_items in clusterized_proposals.items():
    list_samples = [cluster_items]
    #if len(cluster_items) > 892:
    if len(cluster_items) > 30892:
        list_samples = [x for x in chunked_iterator(cluster_items, 892, 446)]
    cluster_title_to_chunks[cluster_title] = list_samples
    print(cluster_title, len(list_samples), len(list_samples[-1]))

In [ ]:
from transformers import AutoTokenizer
model_id = "meta-llama/Llama-3.3-70B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
import numpy as np
max_tokens = 16000
chunks = []
current_token_count = 0

for cluster, cluster_chunks in cluster_title_to_chunks.items():
    text = "\n".join([x['title'] for x in cluster_chunks[0]])
    tokens = tokenizer.encode(text, add_special_tokens=True)
    chunks.append(len(tokens))

chunks[0] = 15873
chunks[5] = 15471
print(chunks, np.mean(chunks))

In [ ]:
# Run sub-cluster detection
cluster_subclusters = {}
for cluster_title, list_samples in tqdm(cluster_title_to_chunks.items()):

    for chunk in list_samples:
        data = "Articles\n - " + "\n - ".join([x['title'] for x in chunk])
        r = model_json(prompt_cluster, data)

        print("=======================")
        print(r)
        cluster_subclusters[cluster_title] = r
json.dump(cluster_subclusters, open(cluster_subclusters_fp, 'w'), indent=2, ensure_ascii=False)

## Classify proposal for each cluster into corresponding sub-group

In [ ]:
cluster_subclusters = json.load(open(cluster_subclusters_fp))
classification_results = []
errors = []
previous_errors = json.load(open(second_level_classified_proposals_errors_fp))
previous_errors_ids = [x['id'] for x in previous_errors]
for cluster_title, cluster_items in clusterized_proposals.items():
    sub_clusters = cluster_subclusters[cluster_title]
    sub_cluster_names = [x['name'] for x in sub_clusters]

    prompt_classify = """Your task is to find the most appropriate class for a given text.
    In what follows you can find the set of classes together with their definition.
    You can choose one of the following classes:
    %s
    Output a JSON with the following structure, do not output anything else.
    Before outputin check the JSON is valid. Correct it if it is not

    Output:
        class: str # the class corresponding to the given text.
    """
    for sub_cluster in sub_clusters:
        prompt_classify %= (f"- {sub_cluster['name']}: {sub_cluster['summary']}\n%s")
    prompt_classify %= ""

    # Run classification
    for idx, item in enumerate(tqdm(cluster_items, desc=f"Processing {cluster_title}")):
        #if item['id'] in previous_errors_ids:
        try:
            title = item['title']
            r = model_json(prompt_classify, title)

            if r['class'] in sub_cluster_names:
                classification_results.append({
                    'title': title,
                    'second_level_cluster': r['class'],
                    'first_level_cluster': cluster_title,
                    'language': item['language'],
                    'UGC': item.get('UGC', None),
                    'id': item['id']
                })
            else:
                print("Error")
                errors.append({
                    'title': title,
                    'id': item['id']
                })
        except:
            print("Error")
            errors.append({
                'title': title,
                'id': item['id'],
                'non_clustering_error': True
                })
        if idx % 500 == 0:
            json.dump(classification_results, open(second_level_classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
            json.dump(errors, open(second_level_classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)

    # End loop on cluster items
# End loop on clusters
json.dump(classification_results, open(second_level_classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(errors, open(second_level_classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)

second_level_clusterized_proposals = {}

for proposal in classification_results:
    if proposal['first_level_cluster'] not in second_level_clusterized_proposals:
        second_level_clusterized_proposals[proposal['first_level_cluster']] = {}
    if proposal['second_level_cluster'] not in second_level_clusterized_proposals[proposal['first_level_cluster']]:
        second_level_clusterized_proposals[proposal['first_level_cluster']][proposal['second_level_cluster']] = []

    second_level_clusterized_proposals[proposal['first_level_cluster']][proposal['second_level_cluster']].append({
        'id': proposal['id'],
        'title': proposal['title'],
        'language': proposal['language'],
        'UGC': proposal['UGC'],
    })

json.dump(second_level_clusterized_proposals, open(second_level_clusterized_proposals_fp, 'w'), indent=2, ensure_ascii=False)

## Analyse clusters

In [ ]:
anaysis_results = []
proposal_embeddings_fp = "../out/proposal_embeddings.json"
proposal_embeddings = json.load(open(proposal_embeddings_fp))

### Compute Silhouette

In [ ]:
# Compute Silhouette
second_level_clusterized_proposals = json.load(open(second_level_clusterized_proposals_fp))

experiment_proposals = len(set([p['id'] for p in json.load(open(second_level_classified_proposals_fp))]))
total_proposals = len(proposal_embeddings)

from sklearn.metrics import silhouette_samples, silhouette_score

for embedding_type in ["e5", "qwen", "bert", "qwen_128", "laser"]:
    embeddings = []
    labels = []
    idx = 0
    for first_level_cluster in second_level_clusterized_proposals:
        for second_level_cluster, cluster_proposals in second_level_clusterized_proposals[first_level_cluster].items():
            for proposal in cluster_proposals:
                embeddings.append(proposal_embeddings[proposal['id']][f"embedding_{embedding_type}"])
                labels.append(idx)
            idx += 1
    avg_silhouette = silhouette_score(embeddings, labels)
    sample_silhouette_values = silhouette_samples(embeddings, labels)

    # results
    anaysis_results.append({
        "algorithm": "llm-based",
        "filtering": "no",
        "total sentences": experiment_proposals,
        "%sent.": "{:.2f}".format(float(experiment_proposals)/float(total_proposals)*100) + "\%",
        "total clusters": len(clusterized_proposals),
        "embedding type": embedding_type,
        "avg. silhouette": "{:.3f}".format(avg_silhouette)
    })

    print("=== Silhouette for embedding type", embedding_type, "===")
    print("Average silhouette score :", avg_silhouette)
    for first_level_cluster in second_level_clusterized_proposals:
        for idx, cluster_name in enumerate(second_level_clusterized_proposals[first_level_cluster].keys()):
            print(f"Silhouette for cluster {cluster_name}: ", np.mean(sample_silhouette_values[np.array(labels) == idx]))


### Compute centroids and summaries with centroids

In [ ]:
from collections import Counter
import numpy as np
from scipy import spatial


def summarise_clusters_proximity_to_centroids(labels, all_txt, all_emb, all_lang, top_n=2, verbose=False):

    map_cl_data = {}

    centroids = []

    for cl_id, txt, emb, lang in zip(labels, all_txt, all_emb, all_lang):

        if cl_id not in map_cl_data:
            map_cl_data[cl_id] = []
        map_cl_data[cl_id].append([txt, emb, lang])

    for cl_id, data in map_cl_data.items():
        cl_embs = [x[1] for x in data]

        avg_emb = np.array(cl_embs).mean(axis=0).tolist()
        centroids.append(avg_emb)

        for d in data:
            dist = 1. - spatial.distance.cosine(avg_emb, d[1])

            d.append(dist)

    list_clusters = []
    i=0

    for cl_id, data in map_cl_data.items():

        data = sorted(data, key=lambda x: x[3], reverse=True)

        count_lang = Counter([x[2] for x in data])

        if verbose:
            print("* CLUSTER >>", cl_id, "<<", "len:", len(data), count_lang)

        clust_sum = {
            "id": cl_id,
            "count": len(data),
            "count_lang": count_lang.most_common(),
            "top": [],
            "mid": [],
            "low": [],
            "top_by_lang": {},
            "all": [],
            "centroid_vector": centroids[i]
        }
        i=i+1

        if verbose:
            print("top")

        for txt, emb, lang, dist in data[:top_n]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["top"].append([txt, dist])

        if verbose:
            print("mid")

        half_top_n = int(top_n / 2.) if top_n > 2 else 1

        for txt, emb, lang, dist in data[int(len(data)/2.)-half_top_n:int(len(data)/2.)+half_top_n]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["mid"].append([txt, dist])


            if verbose:
                print("low")

        for txt, emb, lang, dist in data[-top_n:]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["low"].append([txt, dist])


        if verbose:
            print("    ------")

        for lang in count_lang.keys():
            data_lang = [x for x in data if x[2] == lang]

            for txt, emb, _, dist in data_lang[:2]:
                if verbose:
                    print("   ", lang, ":", "%.2f" % dist, "-", txt)

                if lang not in clust_sum["top_by_lang"]:
                    clust_sum["top_by_lang"][lang] = []
                clust_sum["top_by_lang"][lang].append([txt, dist])

        if verbose:
            print("    ------")

        for txt, emb, lang, dist in data:
            clust_sum["all"].append([txt, dist, lang])

        list_clusters.append(clust_sum)

    return list_clusters

In [ ]:
embedding_types = ["qwen_128", "qwen", "bert", "e5", "laser"]
for embedding_type in embedding_types:
    # Build lists to compute centroids
    labels = []
    proposal_ids = []
    embeddings = []
    languages = []

    for first_level_cluster in second_level_clusterized_proposals:
        for second_level_cluster, cluster_proposals in second_level_clusterized_proposals[first_level_cluster].items():
            for proposal in cluster_proposals:
                if proposal.get('UGC') is not None:
                    labels.append(second_level_cluster)
                    proposal_ids.append(proposal['id'])
                    embeddings.append(proposal_embeddings[proposal['id']][f'embedding_{embedding_type}'])
                    languages.append(proposal['language'])

    cluster_summaries = summarise_clusters_proximity_to_centroids(labels, proposal_ids, embeddings, languages)
    json.dump(cluster_summaries, open(cluster_summaries_fp.replace(".json", f"_{embedding_type}.json"), 'w'), ensure_ascii=False)

### Filter cluster with centroids and recompute metrics

In [ ]:
from collections import defaultdict
threshold_map = {
    "qwen_128": 0.76,
    "qwen": 0.70,
    "bert": 0.62,
    "e5": 0.85,
    "laser": 0.65
}
experiment_proposals = defaultdict(list)
for embedding_type in embedding_types:
    print("Filtering clusters with ", embedding_type)
    cluster_summaries = json.load(open(cluster_summaries_fp.replace(".json", f"_{embedding_type}.json")))
    # Filter clusters
    cluster_centroids = {}
    for summary in cluster_summaries:
        cluster_centroids[summary['id']] = summary['centroid_vector']

    filtered_clusters = defaultdict(list)
    FILTERING_THRESHOLD = threshold_map[embedding_type]
    for first_level_cluster in second_level_clusterized_proposals:
        for second_level_cluster, cluster_proposals in second_level_clusterized_proposals[first_level_cluster].items():
            cluster_centroid = cluster_centroids[second_level_cluster]

            for proposal in cluster_proposals:
                proposal_embedding = proposal_embeddings[proposal['id']][f"embedding_{embedding_type}"]
                sim = 1. - spatial.distance.cosine(cluster_centroid, proposal_embedding)
                if sim >= FILTERING_THRESHOLD:
                    filtered_clusters[second_level_cluster].append(proposal)
    json.dump(filtered_clusters, open(filtered_clusters_fp.replace(".json", f"_{embedding_type}.json"), "w"), ensure_ascii=False)
    
    for cluster_name, cluster_proposals in filtered_clusters.items():
        experiment_proposals[embedding_type].append(len(cluster_proposals))


In [ ]:
# Compute Silhouette
from sklearn.metrics import silhouette_samples, silhouette_score

for embedding_type in ["e5", "qwen", "bert", "qwen_128", "laser"]:
    embeddings = []
    labels = []
    filtered_clusters = json.load(open(filtered_clusters_fp.replace(".json", f"_{embedding_type}.json")))
    for idx, cluster_proposals in enumerate(filtered_clusters.values()):
        for proposal in cluster_proposals:
            embeddings.append(proposal_embeddings[proposal['id']][f"embedding_{embedding_type}"])
            labels.append(idx)

    avg_silhouette = silhouette_score(embeddings, labels)
    sample_silhouette_values = silhouette_samples(embeddings, labels)

    # results
    anaysis_results.append({
        "algorithm": "llm-based",
        "filtering": "yes",
        "total sentences": sum(experiment_proposals[embedding_type]),
        "%sent.": "{:.2f}".format(float(sum(experiment_proposals[embedding_type]))/float(total_proposals) * 100) + "\%",
        "total clusters": len(clusterized_proposals),
        "embedding type": embedding_type,
        "avg. silhouette": "{:.3f}".format(avg_silhouette)
    })

    print("=== Silhouette for embedding type", embedding_type, "===")
    print("Average silhouette score :", avg_silhouette)
    for idx, cluster_name in enumerate(filtered_clusters):
        print(f"Silhouette for cluster {cluster_name}: ", np.mean(sample_silhouette_values[np.array(labels) == idx]))


In [ ]:
json.dump(anaysis_results, open(analysis_results_fp, 'w'), indent=2)

In [ ]:
import json
import pandas as pd
import os

out_folders = ["out", "out_chunk_size_16k", "out_100"]

setting = {
    "out": "19.5k-10clusters",
    "out_chunk_size_16k": "16k-10clusters",
    "out_100": "16k-100clusters",
}
results = []
for out_folder in out_folders:
    result_fp = f"./{out_folder}/5.results.json"
    experiment_results = json.load(open(result_fp))
    for x in experiment_results:
        x['setting'] = setting[out_folder]
    results.extend(experiment_results)

    if os.path.exists(f"./{out_folder}/7.second_level_results.json"):
        experiment_results = json.load(open(f"./{out_folder}/7.second_level_results.json"))
        for x in experiment_results:
            x['setting'] = '16k-10clusters-second level'
        results.extend(experiment_results)

    print(out_folder, len(results))
df = pd.DataFrame(results)
print(df)
print(df.to_latex(index=False))